# Session 4: Structured Errors and Error Propagation
**CCAR-F task statements:** 2.2 (structured error responses for tools) and 5.3 (error propagation across agents) · **Domains:** D2 Tool Design and MCP (18%), D5 Context and Reliability (15%)
**Duration:** 60 minutes · **Extends:** Exercise 1, the Northwind support agent (step 3: structured errors)
**Pre-work:** none of the courses cover this, so the docs are the source: MCP specification *Tools → Error Handling*; Claude docs *Handle tool calls* (`is_error`, instructive error messages); Anthropic Engineering, *Writing effective tools for agents* (error responses).

By the end of this session you can:
1. Design structured error responses that let an agent choose **retry, explain, or escalate**.
2. Separate retryable from non-retryable failures, and tell a real failure from a **valid empty result**.
3. Design how a worker/subagent reports failure to a coordinator: recover locally, propagate only what is unresolved, with context.

| Time | Section | Mode |
|---|---|---|
| 6 min | 1. Setup and recap of Session 3 | run + discuss |
| 6 min | 2. Essential concepts + the "Operation failed" baseline | short explanation + live demo |
| 6 min | 3. Exercise A: design four error payloads | design |
| 15 min | 4. Build the structured error layer (TODOs 1 to 5) | build + offline checks |
| 9 min | 5. Live: retry, explain, or escalate? (Exercise B, TODO 6) | predict + test |
| 12 min | 6. Errors across agents (TODO 7) | build + test |
| 6 min | 7. Exam-style scenarios and wrap-up | reason aloud |

> **Instructor note.** Live cells make real API calls (about 25 short Haiku calls in total). Run sections 2 and 5 once *before* the learner joins so you have transcripts to fall back on. Everything in section 4 is offline and needs no key.

## 1. Setup and recap

Same project, new folder. What changed in `northwind/`:

* `errors.py` (new): the exceptions a real backend raises: `ServiceTimeout`, `InvalidInput`, `NotFound`, `PolicyViolation`, `PermissionDenied`.
* `backend.py`: functions now **raise** those instead of returning `{"error": ...}`. Two new records: order **12455** (a $649 tent, above the $500 automatic-refund limit) and **Sam Okafor**, a real customer with **no orders**. `inject_fault()` makes outages reproducible.
* `common.py`: `run_tool` is the naive catch-all most teams write first. Every failure becomes `"Operation failed"`.
* `tool_defs.py`: the Session 3 reference descriptions, so we start from good tool selection. `agent_loop.py`: the Session 3 loop (`tools`, `system`, `tool_choice`).
* `order_history.py`, `error_demo.py`, `answer_keys.py`, `test_errors_offline.py`: today's helpers and checks.

In [1]:
import os, sys, json, copy, time
sys.path.insert(0, "northwind")

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in the environment that launched Jupyter, then restart the kernel."

import common, backend
from common import MODEL, SYSTEM, TOOLS, client, pick_tools
from errors import ServiceTimeout, InvalidInput, NotFound, PolicyViolation, PermissionDenied
from agent_loop import run_agent
from error_demo import run_case, trace
from test_loop_offline import check_loop, report
from test_tool_choice_offline import check_s3
import test_errors_offline as t4

common.run_tool, common.as_tool_result = common.run_tool_naive, common.as_tool_result_plain   # start naive
print("Model:", MODEL)
print("Tools:", [t["name"] for t in TOOLS])

# What the backend does now when something is wrong:
for call in [lambda: backend.lookup_order("99999"),
             lambda: backend.lookup_order("12-34"),
             lambda: backend.process_refund("12455", 649, "torn")]:
    try:
        call()
    except Exception as e:
        print(f"  {type(e).__name__:<16} {e}")

Model: claude-haiku-5-5
Tools: ['lookup_order', 'get_customer', 'track_shipment', 'process_refund']
  NotFound         No order with number 99999 exists
  InvalidInput     '12-34' is not a valid order number
  PolicyViolation  Refund of 649.00 is above the 500.00 automatic-refund limit


### Recap: Session 3 in five lines

| Idea | What to remember |
|---|---|
| Descriptions drive selection | Name, description and schema are all Claude sees. Say what it returns, when to use it, when **not** to (and which tool instead), input formats, edge cases. |
| Fix ladder | Describe first, then rename, split, or narrow. Consolidate near-identical verbs on one resource. |
| Keyword trap | A system-prompt rule that names a tool by keyword can override good descriptions. Read the prompt when selection is still wrong. |
| Least privilege | Each agent gets only its role's tools, plus a narrow cross-role tool if needed (`verify_fact`). Enforce it in code too. |
| `tool_choice` | `auto`, `any`, forced `tool`, `none`. Forcing guarantees the **first** call only; follow-up turns go back to `auto`. |

**Ask the learner (2 minutes):**
1. Your agent keeps sending shipping questions to `lookup_order` even after you rewrote the descriptions. What do you check next?
2. Why did the Session 3 loop refuse a `process_refund` call from the status agent in code, when that tool was never sent?
3. Homework transcript: what did the agent tell the customer about order `99999`? Could it tell whether the order did not exist or the system was down?

<details><summary>Answers</summary>

1. The system prompt, for a rule that ties a keyword to a tool (official sample Q2 reasoning: root cause before adding layers).
2. Least privilege belongs in code as well as in what you send. Defence in depth.
3. Usually something vague ("I couldn't find that order"). With the old string error it could not distinguish *not found* from *service down*. That gap is today's topic.
</details>

## 2. Essential concepts

### Why "Operation failed" blocks recovery
An agent can only recover from what it can see. If a timeout, a typo'd order number, a policy refusal and a missing permission all come back as `"Operation failed"`, the agent must guess. It will retry things that can never succeed, tell customers to "try later" when they mistyped a number, and apologise vaguely when it should escalate.

### Where the error flag lives
* **MCP** has two mechanisms. *Protocol errors* (JSON-RPC: unknown tool, bad arguments) and *tool execution errors*, reported **inside the tool result** with `"isError": true` "so the model can see them": API failures, invalid input data, business-logic errors.
* **Claude API**: the same idea is `"is_error": true` on the `tool_result` block. The docs ask for *instructive* messages: what went wrong and what to try next, not `"failed"`.
* `errorCategory`, `isRetryable` and `customer_message` are **not** protocol fields. They are a convention you put in the content so the agent (and your code) can act. The exam guide uses these names.

### Four categories, four different next moves

| Category | Northwind example | Retryable? | What should happen |
|---|---|---|---|
| **transient** | order service timed out | yes | retry *locally* with backoff; if still failing, tell the customer it's temporary |
| **validation** | order `99999` does not exist; `12-34` is malformed | no | fix the input if obvious, otherwise **ask** the customer |
| **business** | refund $649 > $500 automatic limit | no | **explain** with a customer-friendly message, offer the allowed path (approval) |
| **permission** | status agent asked to refund; credential lacks scope | no | **hand off / escalate**, never retry |

### Not every "nothing" is an error
A customer with **no orders**, or an order with **no tracking number yet**, is a correct answer. Mark it as an error and the agent apologises for a fault that never happened. Mark a real failure as an empty success and the agent confidently tells a customer they have no orders.

In [2]:
# Baseline: the naive layer on three very different failures.
REFUND_TOOLS = pick_tools("get_customer", "lookup_order", "process_refund")

run_case("Where is order 99999?", run_agent, label="A. order does not exist")
run_case("Where is order 12345?", run_agent, faults=[("track_shipment", "timeout", 99), ("lookup_order", "timeout", 99)],
         label="B. order service is down")
run_case("Order 12455: the tent ripped on the first night. Please refund the full 649 dollars.", run_agent,
         tools=REFUND_TOOLS, label="C. refund above the automatic limit")


=== A. order does not exist
USER: Where is order 99999?
  call 1: track_shipment({"order_id": "99999"}) -> ERROR 'Operation failed'
AGENT: I couldn't retrieve shipping information for order 99999. The system returned an error, which usually means the order number isn't in our system or was mistyped.

Could you double-check the number? If you're not sure of it, send me the email address on your account and I'll look up your orders.

=== B. order service is down
USER: Where is order 12345?
  call 1: track_shipment({"order_id": "12345"}) -> ERROR 'Operation failed'
  call 2: track_shipment({"order_id": "12345"}) -> ERROR 'Operation failed'
AGENT: I'm sorry, I couldn't get the shipping status for order 12345. The tracking system returned an error twice in a row.

Here's what you can do:
- **Try again in a few minutes.** This is often a temporary issue.
- **Check the order details.** I can look up the items and status for order 12345 if that helps.
- **Contact us again later.** If it still

('I\'m sorry, but I wasn\'t able to process the refund for order 12455. The system returned an "Operation failed" error twice, so no money has been moved.\n\nI\'ve confirmed the order details: the Summit 4-season expedition tent (SKU TENT-4S), $649.00 USD, delivered and owned by your account. The full amount is eligible for refund once the system is working.\n\nPlease try again in a little while. If it still fails, contact our support team directly and mention order 12455 and the tent ripping on the first night, so they can resolve it quickly. I\'m sorry for the trouble with the tent.',
 [{'role': 'user',
   'content': 'Order 12455: the tent ripped on the first night. Please refund the full 649 dollars.'},
  {'role': 'assistant',
   'content': [ThinkingBlock(signature='CAQSxQUKEAgTGAI4AUIIdGhpbmtpbmcSDP/iMAKy/x4SMMYuPhoMDMupRPzc1ppo/J2uIjDpGMQWrh+mCztg0k68pcCUskE/ex8I1hudusk1Zztk7EcH/GU4pBt/149rLIkESXkq4gT9+7drIJNLy1oQIenkICHRudMFCHlNm6SoTKW+9EmJN4byL0YPAnvuTbYodSpJdVvEOUndQYg5/lr410Rf

**Observe and explain:**
1. Read the three tool results. Can *you* tell which case was which from what Claude received?
2. In C, did the agent tell the customer the refund needs approval? Did it retry, or try a smaller amount?
3. In A, did it ask the customer to check the number, or suggest "try again later"?

<details><summary>What usually happens</summary>

All three results are the identical string `{"error": "Operation failed"}`. Claude guesses: it may retry C (and sometimes tries a smaller refund, which is a policy workaround), tell A to "try again later" when the order simply does not exist, and treat B like A. None of that is the model's fault. The interface gave it nothing to reason with.
</details>

## 3. Exercise A: design four error payloads

Before writing code, design what Claude should receive in each case (the curriculum's practice task). Fill in the four dicts. Every payload needs:

* `isError`, `errorCategory`, `isRetryable`, and a `message` written **for the agent**: what happened and what to do next;
* `customer_message` where the agent should pass wording to the customer (policy and permission cases);
* `details`: machine-readable context (attempts, field and expected format, policy code and limit, suggested action).

Think about **who reads each field**: `message` is for Claude, `customer_message` is for the customer, `details` is for code (and logs).

In [ ]:
DESIGN = {
    "timeout": {            # track_shipment: order service did not answer in 5 s, 3 attempts made
        "isError": True,
        "errorCategory": "TODO",
        "isRetryable": None,
        "message": "TODO",
        "details": {},
    },
    "bad_order_id": {       # lookup_order("99999"): well-formed but no such order
        "isError": True,
        "errorCategory": "TODO",
        "isRetryable": None,
        "message": "TODO",
        "details": {},
    },
    "refund_over_limit": {  # process_refund 649.00 on order 12455; automatic limit is 500.00
        "isError": True,
        "errorCategory": "TODO",
        "isRetryable": None,
        "message": "TODO",
        "customer_message": "TODO",
        "details": {},
    },
    "no_permission": {      # the read-only status agent tried process_refund
        "isError": True,
        "errorCategory": "TODO",
        "isRetryable": None,
        "message": "TODO",
        "customer_message": "TODO",
        "details": {},
    },
}

from answer_keys import review_design
review_design(DESIGN)

**Why did you choose this approach?**
1. Why is `bad_order_id` *not retryable*, when a timeout on the very same tool is?
2. The refund-over-limit payload says `isRetryable: false`. What could go wrong if it said `true`?
3. Why give the business error a separate `customer_message` instead of letting Claude phrase the policy itself?

<details><summary>Answers</summary>

1. Retrying the same input produces the same answer: the order still won't exist. Only the *customer* can supply the fix. A timeout is about the system's state, which changes on its own.
2. The agent loops on a refusal, or "retries" with a smaller amount (two $324.50 refunds), which is working around a policy. Retryability is a safety property, not a convenience.
3. Policy wording is legal/business language that should be exact and consistent. The agent passes it on; it should not improvise what the policy says.
</details>

## 4. Build the structured error layer

Architecture: the **backend raises**, the **tool layer translates**, Claude **decides**.

```text
Claude ──tool_use──▶ run_tool_structured ──▶ call_with_retry ──▶ backend function
                         │   (TODO 3)             (TODO 2)          raises ServiceTimeout /
                         │                                          InvalidInput / NotFound /
                         ◀── structured_error(...) (TODO 1) ◀──────  PolicyViolation / PermissionDenied
Claude ◀──tool_result + is_error (TODO 4)
```

Transient failures are retried **locally** first. Claude only sees what is still unresolved.

### TODO 1: `structured_error`
Return the payload dict. Only `transient` is retryable by default; `retryable=` overrides. Put `customer_message` in only when given, and any extra keyword arguments under `details`. Reject unknown categories with `ValueError` (a typo like `"buisness"` would silently mislead the agent).

In [ ]:
CATEGORIES = ("transient", "validation", "business", "permission")

def structured_error(category, message, *, retryable=None, customer_message=None, **details):
    # TODO 1: validate category, choose isRetryable (default by category, or the override),
    #         build {"isError": True, "errorCategory": ..., "isRetryable": ..., "message": ...},
    #         add "customer_message" only if given and "details" only if there are any.
    raise NotImplementedError("TODO 1")

def is_error(result):
    return isinstance(result, dict) and result.get("isError") is True

report(t4.check_structured_error(structured_error))

### TODO 2: `call_with_retry`
Call `fn(**tool_input)`. On `ServiceTimeout`, sleep `base_delay * 2**(attempt-1)` and try again, up to `max_attempts`. Return `(result, attempts)`. When attempts run out, record them (`exc.details["attempts"] = attempt`) and **re-raise**. Any other exception is re-raised immediately: retrying a bad order number cannot help.

In [ ]:
def call_with_retry(fn, tool_input, *, max_attempts=3, base_delay=0.2, sleep=time.sleep):
    # TODO 2: loop attempt = 1..max_attempts
    #           try: return fn(**tool_input), attempt
    #           except ServiceTimeout: last attempt -> record attempts and raise; else sleep with backoff
    #         (InvalidInput, NotFound, ... are NOT caught here: they propagate immediately)
    raise NotImplementedError("TODO 2")

report(t4.check_retry(call_with_retry))

### TODO 3: `run_tool_structured`
Look the tool up in `common.TOOL_FUNCTIONS` (unknown name → `validation`). Call it through `call_with_retry`. On success return the result **unchanged**, including empty lists. Map each exception to a category with an instructive message:

| Exception | Category | The message should tell Claude... |
|---|---|---|
| `ServiceTimeout` (after retries) | transient | it was already retried N times; don't guess; tell the customer it's temporary |
| `InvalidInput`, `NotFound` | validation | don't retry the same value; fix it if obvious, else ask (or `get_customer`) |
| `PolicyViolation` | business | don't retry or work around it; use `customer_message`; offer the alternative |
| `PermissionDenied` | permission | don't retry; hand off / escalate |
| `TypeError` (bad arguments) | validation | check the input schema |

Every exception carries `exc.message` and `exc.details`; `PolicyViolation` also has `exc.customer_message`. Pass `exc.details` through so codes and limits reach the payload.

In [ ]:
def run_tool_structured(name, tool_input, *, max_attempts=3, base_delay=0.2, sleep=time.sleep):
    fn = common.TOOL_FUNCTIONS.get(name)
    # TODO 3a: unknown tool -> structured_error("validation", ...)
    try:
        result, attempts = call_with_retry(fn, tool_input, max_attempts=max_attempts,
                                           base_delay=base_delay, sleep=sleep)
        return result
    except ServiceTimeout as exc:
        raise NotImplementedError("TODO 3b: transient error with details attempts=...")
    # TODO 3c: InvalidInput and NotFound      -> validation
    # TODO 3d: PolicyViolation               -> business, with customer_message=exc.customer_message
    # TODO 3e: PermissionDenied              -> permission, suggested_action="escalate_to_human"
    # TODO 3f: TypeError                     -> validation ("check the input schema")

report(t4.check_run_tool(run_tool_structured))

### TODO 4: `as_tool_result` with `is_error`
Same as before, plus `"is_error": True` on the block when the payload is an error. Successful results (including empty ones) carry no flag.

In [ ]:
def as_tool_result(tool_use_block, result):
    block = {"type": "tool_result", "tool_use_id": tool_use_block.id, "content": json.dumps(result)}
    # TODO 4: mark the block as an error when the payload is one
    return block

report(t4.check_tool_result(as_tool_result))

### TODO 5: a structured permission error inside the loop
Below is the Session 3 loop. When Claude asks for a tool **outside this agent's scope**, it still returns a bare string. Replace it with a structured `permission` error (not retryable, a `customer_message`, and `details` naming the tool, the allowed tools and `suggested_action: "handoff"`). The backend must still never run.

In [ ]:
def run_agent(user_message, client=client, max_iterations=15, verbose=True,
              tools=None, system=None, tool_choice=None):
    tools = TOOLS if tools is None else tools
    system = SYSTEM if system is None else system
    allowed = {t["name"] for t in tools}
    messages = [{"role": "user", "content": user_message}]

    for step in range(1, max_iterations + 1):
        request = dict(model=MODEL, max_tokens=1024, system=system, tools=tools, messages=messages)
        if tool_choice is not None and step == 1:
            request["tool_choice"] = tool_choice
        response = client.messages.create(**request)
        if verbose:
            print(f"[step {step}] stop_reason={response.stop_reason}")

        if response.stop_reason == "end_turn":
            final_text = "".join(b.text for b in response.content if b.type == "text")
            return final_text, messages + [{"role": "assistant", "content": response.content}]

        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue
                if block.name in allowed:
                    output = common.run_tool(block.name, block.input)
                else:
                    output = {"error": f"Tool '{block.name}' is not available to this agent"}   # TODO 5
                if verbose:
                    print(f"          tool {block.name}({block.input}) -> {str(output)[:80]}")
                results.append(common.as_tool_result(block, output))
            messages.append({"role": "user", "content": results})
            continue

        raise RuntimeError(f"Unhandled stop_reason: {response.stop_reason}")

    raise RuntimeError(f"Hit the {max_iterations}-iteration guard rail without end_turn")

print("TODO 5");               report(t4.check_agent_permission(run_agent, as_tool_result))
print("\nSession 2 regression"); report(check_loop(run_agent))
print("\nSession 3 regression"); report(check_s3(run_agent))

Stuck after two attempts on a TODO? Load the reference, read it, run it, re-run that TODO's check. (Uncomment one line. `%load` replaces the cell's content with the file.)

> **Instructor note.** The two classic mistakes: (1) retrying inside `run_tool_structured` *and* inside `call_with_retry` (9 attempts instead of 3; check "attempts=3" catches it); (2) catching `Exception` first, so every case becomes the same category again. Order the `except` clauses from specific to general.

In [ ]:
# %load northwind/solutions/tool_errors_ref.py
# %load northwind/solutions/agent_loop.py

### Wire it in
`agent_loop` and `run_agent` look up `common.run_tool` and `common.as_tool_result` at call time, so two assignments switch the whole agent to the structured layer.

In [ ]:
common.run_tool = lambda name, tool_input: run_tool_structured(name, tool_input)
common.as_tool_result = as_tool_result
print(json.dumps(common.run_tool("lookup_order", {"order_id": "99999"}), indent=2))
print(json.dumps(common.run_tool("get_customer", {"email": "sam.okafor@example.com"}), indent=2))

## 5. Live: retry, explain, or escalate?

### Exercise B: predict first
For each outcome, write what should happen, using: `local` (recovered inside the tool layer, Claude never sees it), `ask` (ask the customer), `not-error` (a normal result), `explain` (explain the policy, offer the alternative), `handoff` (pass to a team/human that can do it), `later` (temporary problem, offer to try again later).

In [ ]:
EX_B = {
    1: "",  # track_shipment timed out once, the second attempt worked
    2: "",  # lookup_order("99999")
    3: "",  # lookup_order(" #12345 ")
    4: "",  # process_refund(12455, 649.00): above the $500 automatic limit
    5: "",  # the read-only status agent calls process_refund
    6: "",  # get_customer for sam.okafor@example.com returns order_ids: []
    7: "",  # track_shipment(12399): status processing, carrier and tracking are null
    8: "",  # track_shipment timed out on all 3 attempts
}
from answer_keys import review_ex_b
review_ex_b(EX_B)

Now run the same three failures as the baseline, plus two new ones, with the structured layer. The system prompt is still the plain one: **structure only, no policy yet.**

In [ ]:
run_case("Where is order 99999?", run_agent, label="A. order does not exist")
run_case("Where is order 12345?", run_agent, faults=[("track_shipment", "timeout", 99), ("lookup_order", "timeout", 99)],
         label="B. order service is down (3 local attempts each, then unresolved)")
run_case("Order 12455: the tent ripped on the first night. Please refund the full 649 dollars.", run_agent,
         tools=REFUND_TOOLS, label="C. refund above the automatic limit")
run_case("Order 12400: the lantern is too dim. Please refund the 34 dollars.", run_agent,
         tools=REFUND_TOOLS, faults=[("process_refund", "permission", 1)],
         label="D. the refunds credential lacks the refunds:write scope")
run_case("Hi, I'm sam.okafor@example.com. What have I ordered from you?", run_agent,
         label="E. real customer with no orders")

**Compare with the baseline.** For each case: did the agent's behaviour match your Exercise B prediction? Where it did not, was the *payload* missing something, or did the agent simply not know your *policy*?

### TODO 6: the error-handling policy in the system prompt
Structure tells Claude **what** happened. The system prompt tells it **what we want done about each kind**. Write that policy: one short rule per category, plus partial results. Keep it about behaviour, not about tool names (Session 3's keyword trap).

In [ ]:
SYSTEM_ERRORS = SYSTEM + '''

When a tool result has "isError": true, read errorCategory and act on it:
- transient: TODO
- validation: TODO
- business: TODO
- permission: TODO
If a result has "status": "partial": TODO'''

assert "TODO" not in SYSTEM_ERRORS, "Write the policy (or load the reference: from solutions.prompts import SYSTEM_ERRORS)"
run_case("Order 12455: the tent ripped on the first night. Please refund the full 649 dollars.", run_agent,
         tools=REFUND_TOOLS, system=SYSTEM_ERRORS, label="C again, with the policy")
run_case("Where is order 12345?", run_agent, system=SYSTEM_ERRORS,
         faults=[("track_shipment", "timeout", 99), ("lookup_order", "timeout", 99)], label="B again, with the policy")

### Who retries: the tool layer or Claude?
Turn local retry **off** (`max_attempts=1`) and inject a **single** timeout. Claude now sees a transient error with `isRetryable: true`. Predict: does it call the tool again by itself?

In [ ]:
common.run_tool = lambda name, tool_input: run_tool_structured(name, tool_input, max_attempts=1)
run_case("Where is order 12345?", run_agent, system=SYSTEM_ERRORS,
         faults=[("track_shipment", "timeout", 1), ("lookup_order", "timeout", 1)], label="one timeout, no local retry")
common.run_tool = lambda name, tool_input: run_tool_structured(name, tool_input)   # back to local retry

**Why did you choose this approach?**
1. Why retry transient failures in the tool layer rather than leaving every retry to Claude?
2. When *should* Claude be the one to retry?
3. The policy prompt and the structured payload overlap. Why keep both?

<details><summary>Answers</summary>

1. Local retry is cheap, fast and deterministic: no extra model turn, no tokens, no chance the model decides not to bother. Most transient blips disappear before Claude ever sees them.
2. When the retry needs judgement: after a pause the customer agreed to, with corrected input, or with a different tool (`lookup_order` instead of `track_shipment`). `isRetryable` tells Claude that a retry *can* help; the policy says *when* it should.
3. They do different jobs. The payload states facts about this failure; the prompt states the business's rules for all failures. Without facts the policy cannot be applied; without the policy each failure is improvised.
</details>

## 6. Errors across agents (5.3)

`get_order_history` lets the support agent (the **coordinator**) ask a **worker** to summarise every order on an account. The worker makes many backend calls on its own. In Session 5 it becomes a real Claude subagent; today it is Python, so we can focus on one question: *when part of its job fails, what should reach the coordinator?*

| Principle | Meaning |
|---|---|
| Recover locally | Transient failures are retried inside the worker. |
| Keep partial results | One failed order must not throw away the others. |
| Propagate only what is unresolved, **with context** | failure type, what was attempted, attempts, retryable?, alternatives |
| Annotate coverage | "2 of 3 orders retrieved; 12400 missing" so the final answer can say what it could not check |
| Anti-patterns | empty result marked as success (suppression); generic "unavailable" status; letting one failure kill the whole workflow |

Three workers that get it wrong are in `order_history.py`. Run each with order **12400** permanently timing out.

In [ ]:
import order_history as oh
from order_history import ORDER_HISTORY_TOOL, use_worker

backend.clear_faults()
backend.inject_fault("lookup_order", "timeout", times=99, when={"order_id": "12400"})
for worker in (oh.worker_silent_empty, oh.worker_generic_status, oh.worker_crash):
    try:
        print(f"{worker.__name__:<22}", worker("alex.rivera@example.com"))
    except Exception as e:
        print(f"{worker.__name__:<22} RAISED {type(e).__name__}: {e}")
backend.clear_faults()

Now see what the **coordinator** tells the customer when the worker suppresses a failure. Here the customer lookup itself times out, so `worker_silent_empty` returns an empty list "successfully".

In [ ]:
HISTORY_TOOLS = TOOLS + [ORDER_HISTORY_TOOL]
use_worker(oh.worker_silent_empty)
run_case("Hi, I'm alex.rivera@example.com. Can you list everything I've ordered?", run_agent,
         tools=HISTORY_TOOLS, system=SYSTEM_ERRORS, faults=[("get_customer", "timeout", 99)],
         label="coordinator + suppressing worker")

Alex has three orders. The agent just told him he has none, confidently, because "empty" and "failed" looked the same. That is the **access failure vs valid empty result** distinction, and the coordinator cannot fix it: the information was destroyed one level down.

### TODO 7: `order_history_worker`
Return a report the coordinator can reason with:

```text
customer lookup fails      -> {"isError": True, "errorCategory": ..., "isRetryable": ..., "status": "failed",
                               "failures": [{"attempted": "get_customer(email='...')", ...}], "orders": [],
                               "alternatives": [...]}
some orders fail           -> {"status": "partial", "orders": [...the good ones...],
                               "failures": [{"order_id", "attempted", "errorCategory", "isRetryable", "attempts", "message"}],
                               "coverage": {"orders_on_account": 3, "retrieved": 2, "missing": ["12400"]},
                               "alternatives": [...]}
everything works / no orders -> {"status": "complete", "orders": [...possibly empty...], "coverage": {...}}
```

Use your `call_with_retry` for every backend call (local recovery). Map exceptions to categories with `CATEGORY_OF` below.

In [ ]:
from backend import get_customer, lookup_order
from errors import NorthwindError

CATEGORY_OF = {ServiceTimeout: "transient", InvalidInput: "validation", NotFound: "validation",
               PolicyViolation: "business", PermissionDenied: "permission"}

def order_history_worker(email, *, max_attempts=3, base_delay=0.2, sleep=time.sleep):
    # TODO 7a: customer = call_with_retry(get_customer, {"email": email}, ...)
    #          on NorthwindError -> return the "failed" report (isError, errorCategory, isRetryable,
    #          failures=[{attempted, errorCategory, isRetryable, attempts, message}], orders=[], alternatives)
    # TODO 7b: for each order_id: call_with_retry(lookup_order, ...); keep successes, record failures
    #          (one failure must NOT stop the loop)
    # TODO 7c: status "partial" if any failures else "complete"; coverage {orders_on_account, retrieved, missing};
    #          alternatives when something is missing. Zero orders is "complete", not an error.
    raise NotImplementedError("TODO 7")

report(t4.check_worker(order_history_worker))

In [ ]:
# Stuck? Uncomment to load the reference into this cell:
# %load northwind/solutions/order_history_ref.py

In [ ]:
use_worker(order_history_worker)
run_case("Hi, I'm alex.rivera@example.com. Can you list everything I've ordered?", run_agent,
         tools=HISTORY_TOOLS, system=SYSTEM_ERRORS, faults=[("lookup_order", "timeout", 99, {"order_id": "12400"})],
         label="coordinator + good worker, one order down")
run_case("Hi, I'm alex.rivera@example.com. Can you list everything I've ordered?", run_agent,
         tools=HISTORY_TOOLS, system=SYSTEM_ERRORS, faults=[("get_customer", "timeout", 99)],
         label="coordinator + good worker, customer lookup down")

**Why did you choose this approach?**
1. Why does the worker return `status: "partial"` as a *success* payload instead of `isError: true`?
2. Why include `attempted` in each failure? Who uses it?
3. Your teammate suggests the worker should simply raise on any failure "so the coordinator knows something went wrong". What do you say?

<details><summary>Answers</summary>

1. The call did useful work: two orders are real, correct data. Marking the whole thing an error invites the coordinator to discard them. The failure is reported *inside* the result, with coverage, so the coordinator can use what exists and say what is missing.
2. The coordinator (to decide whether to retry, ask the customer, or try another tool), and humans reading logs. "lookup_order(order_id='12400') timed out 3 times" is actionable; "unavailable" is not.
3. Raising discards the partial results and, if nothing catches it, ends the whole workflow over one order. Report the failure as data with context; let the coordinator decide.
</details>

## 7. Exam-style scenarios

**Start with official sample Q8** (paraphrased; read the original in the exam guide). In a multi-agent research system, the web-search subagent times out on a complex topic. How should that failure reach the coordinator? Options in spirit: (A) structured error context to the coordinator: failure type, the attempted query, any partial results, alternative approaches; (B) retry with backoff inside the subagent, then return a generic "search unavailable"; (C) catch the timeout and return an empty result marked successful; (D) propagate the exception to a top-level handler that ends the workflow. Reason through each before checking the guide (answer: **A**; B loses the context, C is suppression, D kills the workflow).

Then the five questions below. Take two or three in class; the rest are homework. Root cause first, then the proportionate fix.

---
**Q1 (2.2).** A support agent's tools return `{"error": "Operation failed"}` for timeouts, unknown order numbers and refund-policy denials alike. Logs show the agent retrying policy denials several times and telling customers with mistyped order numbers to "try again later". What is the most effective fix?

A. Add to the system prompt: "If a tool fails, retry up to three times before giving up."
B. Return structured errors with an error category, a retryable flag and a specific description, so the agent can tell the cases apart.
C. Increase the tool timeout so fewer calls fail.
D. Escalate every failed tool call to a human agent.

<details><summary>Answer and reasoning</summary>

**B.** The root cause is that three different situations look identical. Structure lets the agent retry only what can succeed, explain policies, and ask for corrections.
A makes the policy-denial loop worse. C addresses one cause and leaves the other two indistinguishable. D is disproportionate: most of these failures are resolvable by the agent.
</details>

---
**Q2 (5.3).** An order-history subagent catches a database timeout and returns `{"status": "success", "orders": []}`. The coordinator tells the customer they have no orders on file. What is the root cause?

A. The coordinator's prompt does not tell it to double-check empty results.
B. The coordinator should have its own retry logic for subagent calls.
C. The subagent suppressed an access failure as a valid empty result; it should return a structured failure with the error type, the attempted query and any partial results.
D. The coordinator needs a larger model to detect implausible answers.

<details><summary>Answer and reasoning</summary>

**C.** Information was destroyed one level down; nothing upstream can recover it. Fix it where it was lost.
A asks the coordinator to guess at what was erased. B retries a call that reported success. D is the "smarter model" distractor for an interface problem.
</details>

---
**Q3 (2.2).** A customer asks for a $650 refund. Automatic refunds are capped at $500; larger ones need a manager. Which tool result best supports the agent?

A. `{"isError": true, "errorCategory": "transient", "isRetryable": true, "message": "Refund failed"}`
B. `{"isError": true, "errorCategory": "business", "isRetryable": false, "message": "Above the 500.00 automatic limit; do not retry or split; route for approval", "customer_message": "Refunds of this size need approval from our team...", "details": {"code": "REFUND_ABOVE_AUTO_LIMIT", "limit": 500}}`
C. `{"error": "Refund denied"}`
D. A success result with `"status": "pending"` and no explanation.

<details><summary>Answer and reasoning</summary>

**B.** Policy violation: not retryable, with a customer-friendly explanation and the allowed path.
A invites a retry loop on a decision that will not change. C gives no category, reason or next step. D hides the policy: the customer believes the refund is being processed.
</details>

---
**Q4 (5.3).** A research coordinator delegates to four subagents. After local retries, the regulatory-filings subagent still fails with a timeout; the other three return good findings. What should the coordinator do?

A. Abort and report that the research could not be completed.
B. Synthesise from the three successful results and silently omit regulatory filings.
C. Re-run all four subagents until every one succeeds.
D. Synthesise from the three results and annotate the coverage gap: which source type is missing and why, so the reader knows what was not checked.

<details><summary>Answer and reasoning</summary>

**D.** Keep the partial results, make the gap explicit.
A throws away good work over one failure. B presents partial research as complete. C repeats successful work and may never terminate.
</details>

---
**Q5 (2.2).** `search_orders(customer_id, date_range)` returns `isError: true, errorCategory: "validation"` when no orders fall inside the range. Customers who simply had no orders that month are told "something went wrong with your request". What should change?

A. Return a successful result with an empty list (and the query that was run), because no matches is a valid answer.
B. Mark the error `isRetryable: true` so the agent tries again.
C. Tell the agent in the system prompt to ignore validation errors from `search_orders`.
D. Remove `date_range` so the tool always returns something.

<details><summary>Answer and reasoning</summary>

**A.** "No matches" is success. Reserve errors for failures.
B makes the agent repeat a query that will return the same nothing. C teaches the agent to ignore real validation errors too. D changes the tool's purpose to hide a labelling mistake.
</details>

## 8. Save, homework, and next session

The next cell writes your error layer to `northwind/tool_errors.py` (from then on `common.py` uses it automatically), your worker to `northwind/order_history_worker.py`, and your policy prompt to `northwind/prompts.py`. Each one is saved **only if it passes its offline checks**; otherwise the reference is saved and the cell says so. `agent_loop.py` gets the TODO 5 version (the Session 3 version is kept as `agent_loop_s3.py`).

In [ ]:
import inspect, pathlib, shutil, types, importlib.util

def passes(results):
    return all(ok for _, ok, _ in results)

def module_source(funcs):
    # Source for funcs plus the notebook globals they use (literal constants and helper functions).
    seen, consts, sources = set(), {}, []
    def names_in(code):
        out = set(code.co_names)
        for c in code.co_consts:
            if isinstance(c, types.CodeType):
                out |= names_in(c)
        return out
    def visit(f):
        if f.__name__ in seen:
            return
        seen.add(f.__name__)
        for n in sorted(names_in(f.__code__)):
            v = globals().get(n)
            if isinstance(v, types.FunctionType) and v.__module__ == "__main__":
                visit(v)
            elif isinstance(v, (dict, tuple, list, str, int, float)) and n != "SYSTEM":
                try:
                    if eval(repr(v)) == v:
                        consts[n] = repr(v)
                except Exception:  # noqa: BLE001 - not a literal (e.g. a dict keyed by classes)
                    pass
        sources.append(inspect.getsource(f))
    for f in funcs:
        visit(f)
    return "".join(f"{k} = {v}\n" for k, v in consts.items()) + "\n\n" + "\n\n".join(sources)

def save_checked(path, text, check, attr, reference):
    # Write the file, re-import it on its own, re-run its offline check; fall back to the reference.
    path.write_text(text, encoding="utf-8")
    spec = importlib.util.spec_from_file_location(path.stem + "_saved", path)
    mod = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(mod)
        ok = passes(check(getattr(mod, attr)))
    except Exception as e:  # noqa: BLE001
        ok = False
        print(f"  {path.name} did not import on its own: {e}")
    if ok:
        print(f"{path.name}: saved YOURS")
    else:
        shutil.copy(reference, path)
        print(f"{path.name}: saved the REFERENCE (yours fails a check or needs something only the notebook has)")

nw = pathlib.Path("northwind")
IMPORTS = ("import json\nimport time\n\nimport common\n"
           "from errors import InvalidInput, NorthwindError, NotFound, PermissionDenied, PolicyViolation, ServiceTimeout\n")
DOC = chr(34) * 3   # a triple quote, for the module docstrings

layer = (structured_error, is_error, call_with_retry, run_tool_structured, as_tool_result)
if (passes(t4.check_structured_error(structured_error)) and passes(t4.check_retry(call_with_retry))
        and passes(t4.check_run_tool(run_tool_structured)) and passes(t4.check_tool_result(as_tool_result))):
    text = (DOC + "Structured error layer written in the Session 4 notebook.\n"
            "common.py imports run_tool and as_tool_result from here." + DOC + "\n" + IMPORTS + "\n"
            + module_source(layer)
            + "\n\ndef run_tool(name, tool_input):\n    return run_tool_structured(name, tool_input)\n")
    save_checked(nw / "tool_errors.py", text, t4.check_run_tool, "run_tool_structured",
                 nw / "solutions/tool_errors_ref.py")
else:
    shutil.copy(nw / "solutions/tool_errors_ref.py", nw / "tool_errors.py")
    print("tool_errors.py: some checks fail, saved the REFERENCE (compare it with yours)")

worker_text = (DOC + "Order-history worker written in the Session 4 notebook." + DOC + "\n" + IMPORTS
               + "from backend import get_customer, lookup_order\nfrom tool_errors import call_with_retry\n\n"
               "CATEGORY_OF = {ServiceTimeout: 'transient', InvalidInput: 'validation', NotFound: 'validation',\n"
               "               PolicyViolation: 'business', PermissionDenied: 'permission'}\n\n"
               + module_source([order_history_worker]).replace("def call_with_retry(", "def _notebook_call_with_retry("))
save_checked(nw / "order_history_worker.py", worker_text, t4.check_worker, "order_history_worker",
             nw / "solutions/order_history_ref.py")

(nw / "prompts.py").write_text(DOC + "System prompts. SYSTEM_ERRORS written in Session 4 (TODO 6)." + DOC + "\n"
                               + f"SYSTEM_ERRORS = {SYSTEM_ERRORS!r}\n", encoding="utf-8")
print("prompts.py: saved SYSTEM_ERRORS")

loop, backup = nw / "agent_loop.py", nw / "agent_loop_s3.py"
if not backup.exists():
    shutil.copy(loop, backup)
shutil.copy(nw / "solutions/agent_loop.py", loop)
print("agent_loop.py: Session 4 reference (structured permission error). Compare it with your TODO 5.")

### Homework (from the curriculum)

* [ ] **Exercise 1, step 3:** structured errors on **all** tools, then verify the agent **retries transient errors** and **explains business errors**. Run these six live cases with `SYSTEM_ERRORS` and fill a table (case, tool results the agent saw, what it did, correct?):
  1. one timeout on `track_shipment` (should be invisible); 2. persistent timeout; 3. order `99999`; 4. refund of $649 on 12455; 5. refund of $100 on 12400 (above the $34 paid); 6. status agent asked for a refund.
* [ ] Add one sentence to each tool **description** saying how it reports failure (e.g. "Returns isError with errorCategory 'validation' if the order number does not exist"). Re-run the Session 3 selection tests to confirm nothing regressed.
* [ ] Re-do exam Q1, Q2 and Q4 in a few days without the answers.
* [ ] **Pre-work for Session 5:** read the Agent SDK docs on subagents.

### Next session
**Session 5: Multi-agent orchestration and subagent configuration (1.2, 1.3).** Today's worker was Python. Next time it becomes a real subagent with its own prompt and tools, and the coordinator delegates to several in parallel. The propagation rules you wrote today are exactly what those subagents must follow.

> **Instructor note.** Before Session 5, check the homework table. Any row where the agent invented data after an unresolved transient error is a good opening: that is a policy (prompt) gap, not a payload gap.

### References
* MCP specification, Tools → Error Handling: https://modelcontextprotocol.io/specification/2025-06-18/server/tools
* Claude docs, Handle tool calls (`is_error`, instructive error messages): https://platform.claude.com/docs/en/agents-and-tools/tool-use/handle-tool-calls
* Anthropic Engineering, Writing effective tools for agents: https://www.anthropic.com/engineering/writing-tools-for-agents
* Claude docs, Tool use overview: https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview